# CVAE Video Prediction — Cyclical KL (Kaggle T4)

**Setup before running:**
1. Set **Accelerator** to GPU T4 and enable **Internet**.
2. Add the Dance dataset as a notebook input (`Notebook → Add Input → Datasets`). The dataset must contain `train/`, `val/`, `test/` folders with `train_img/train_label`, `val_img/val_label`, `test_img/test_label` inside.
3. **To resume**: zip the prior run's save dir (containing `epoch=N.ckpt` snapshots and `history.json`), upload it as a Kaggle dataset, add it as input, and set `RESUME_DIR` in the config cell. The latest `epoch=N.ckpt` is auto-detected; `history.json` is staged into `SAVE_ROOT` so loss/PSNR curves stay continuous across the resume boundary.
4. After training, download artifacts from the **Output** tab (`epoch=N.ckpt`, `epoch=last.ckpt`, `history.json`, plot PNGs).

In [ ]:
!pip install -q imageio tqdm

In [ ]:
# Clone private repo using a GitHub personal access token
# 1. GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens
# 2. Generate a token with: Repository permissions → Contents → Read-only
# 3. Replace YOUR_TOKEN, YOUR_USERNAME, and YOUR_REPO below
# ⚠️  Do not share this notebook with the token filled in.
#
# rm -rf first: if the kernel was previously run, /kaggle/working/ persists,
# a bare `git clone` silently fails into the existing dir, and `%cd` lands
# in the stale pre-fix copy — exactly how fresh code can appear to not take effect.
!rm -rf /kaggle/working/cvae-vanilla
!git clone -b vanilla-cvae https://YOUR_TOKEN@github.com/YOUR_USERNAME/YOUR_REPO.git /kaggle/working/cvae-vanilla
%cd /kaggle/working/cvae-vanilla
!git log -1 --oneline

In [ ]:
import os

# ── Edit these paths ────────────────────────────────────────────────
DATASET_NAME   = 'lab4-dataset'   # your Dance Kaggle dataset name (contains train/ val/ test/)
DATASET_PATH   = f'/kaggle/input/{DATASET_NAME}'

# Run name — output dir is /kaggle/working/runs/<RUN_NAME>
RUN_NAME       = 'cyclical-kl'
SAVE_ROOT      = f'/kaggle/working/runs/{RUN_NAME}'

# Training hyperparameters (project defaults from README — vanilla cyclical)
BATCH_SIZE       = 8           # T4 16 GB handles bs=8 at 32x64 resolution
TRAIN_VI_LEN     = 16
VAL_VI_LEN       = 630
NUM_EPOCH        = 70
LR               = 1e-3
LR_MIN           = 1e-5
SCHEDULER        = 'cosine'    # 'cosine' or 'multistep'
WEIGHT_DECAY     = 0.0

# Teacher-forcing schedule
TFR              = 1.0
TFR_SDE          = 10
TFR_D_STEP       = 0.1
TFR_D_PERIOD     = 6

# KL annealing — 'Cyclical' | 'Monotonic' | 'None' (case-sensitive)
#   Cyclical:  n_cycle cycles; each ramps 0→kl_max for kl_anneal_ratio of its length, then plateau at kl_max
#   Monotonic: linear ramp 0→kl_max over kl_anneal_cycle epochs, then plateau
#   None:      constant beta = kl_max
KL_ANNEAL_TYPE   = 'Cyclical'
KL_ANNEAL_CYCLE  = 4             # Cyclical: # of cycles; Monotonic: ramp length in epochs
KL_ANNEAL_RATIO  = 0.4           # Cyclical only — fraction of each cycle spent ramping
KL_MAX           = 1.0           # peak β; 1.0 is appropriate against the N(0, I) prior

PER_SAVE         = 2
NUM_WORKERS      = 2             # Kaggle gives 2 vCPU

# Resume — upload prior save_root as a Kaggle dataset, add as input, set this
RESUME_DIR       = ''            # e.g. '/kaggle/input/cvae-vanilla-snapshot/'
RESET_OPTIM      = False         # with --resume: rebuild optim+scheduler from current args (covers remaining epochs)
# ────────────────────────────────────────────────────────────────────────

PROJECT_DIR    = '/kaggle/working/cvae-vanilla'
DATASET_SYMLNK = f'{PROJECT_DIR}/dataset'
RUNS_SYMLINK   = f'{PROJECT_DIR}/runs'

os.makedirs(SAVE_ROOT, exist_ok=True)

# Dataset — symlink from input dataset into the repo's expected ./dataset path
if os.path.lexists(DATASET_SYMLNK) and os.path.islink(DATASET_SYMLNK):
    os.unlink(DATASET_SYMLNK)
if not os.path.lexists(DATASET_SYMLNK):
    os.symlink(DATASET_PATH, DATASET_SYMLNK)

# Runs dir — symlink so saves go to /kaggle/working/runs/ and appear in Output
if os.path.lexists(RUNS_SYMLINK) and os.path.islink(RUNS_SYMLINK):
    os.unlink(RUNS_SYMLINK)
if not os.path.lexists(RUNS_SYMLINK):
    os.symlink('/kaggle/working/runs', RUNS_SYMLINK)

print(f'Dataset:     {DATASET_PATH}')
print(f'Save root:   {SAVE_ROOT}')

# ── Resume staging ──────────────────────────────────────────────────
# If RESUME_DIR is set, copy the latest epoch=N.ckpt + history.json
# from there into SAVE_ROOT before training. Required so that:
#   - load_checkpoint() finds the ckpt
#   - History reads history.json (continuous loss/PSNR curves)
import shutil, glob, re

RESUME_CKPT = ''
if RESUME_DIR:
    if not os.path.isdir(RESUME_DIR):
        raise FileNotFoundError(f'RESUME_DIR not a directory: {RESUME_DIR}')
    epoch_ckpts = glob.glob(os.path.join(RESUME_DIR, 'epoch=*.ckpt'))
    # exclude epoch=last.ckpt — we want the highest numbered snapshot
    numbered = [p for p in epoch_ckpts if re.search(r'epoch=(\d+)\.ckpt$', p)]
    if not numbered:
        raise FileNotFoundError(f'No epoch=N.ckpt files in {RESUME_DIR}')
    latest = max(numbered, key=lambda p: int(re.search(r'epoch=(\d+)\.ckpt$', p).group(1)))
    print(f'\nStaging resume artifacts from {RESUME_DIR}:')
    for src in [latest, os.path.join(RESUME_DIR, 'history.json')]:
        name = os.path.basename(src)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(SAVE_ROOT, name))
            print(f'  staged  {name}')
        else:
            print(f'  WARN    {name} not found in RESUME_DIR — skipping')
    RESUME_CKPT = os.path.join(SAVE_ROOT, os.path.basename(latest))
    print(f'  resume  {RESUME_CKPT}')

In [ ]:
import torch

print('=== Environment ===')
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
    print(f'VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\n=== Paths ===')
checks = {
    'Dataset root':      DATASET_PATH,
    'train/train_img':   f'{DATASET_PATH}/train/train_img',
    'train/train_label': f'{DATASET_PATH}/train/train_label',
    'val/val_img':       f'{DATASET_PATH}/val/val_img',
    'val/val_label':     f'{DATASET_PATH}/val/val_label',
    'Save root':         SAVE_ROOT,
    'Dataset symlink':   DATASET_SYMLNK,
    'Runs symlink':      RUNS_SYMLINK,
}
all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ MISSING'
    print(f'  {status}  {name}: {path}')
    if not exists:
        all_ok = False

print('\n=== Dataset ===')
tr_img = f'{DATASET_PATH}/train/train_img'
va_img = f'{DATASET_PATH}/val/val_img'
if os.path.exists(tr_img):
    print(f'  Train frames: {len(os.listdir(tr_img))}')
if os.path.exists(va_img):
    print(f'  Val frames:   {len(os.listdir(va_img))}')

if RESUME_CKPT:
    print('\n=== Resume artifacts (staged in SAVE_ROOT) ===')
    for name in ['history.json', os.path.basename(RESUME_CKPT)]:
        p = os.path.join(SAVE_ROOT, name)
        ok = os.path.exists(p)
        print(f"  {'OK   ' if ok else 'MISS '} {name}")
        if name == os.path.basename(RESUME_CKPT) and not ok:
            all_ok = False

print(f"\n{'✓ All checks passed — ready to train.' if all_ok else '✗ Fix missing paths above before training.'}")

In [ ]:
resume_flags = ''
if RESUME_CKPT:
    resume_flags = f'--ckpt_path {RESUME_CKPT} --resume'
    if RESET_OPTIM:
        resume_flags += ' --reset_optim'

!python Trainer.py \
    --DR              {DATASET_PATH} \
    --save_root       {SAVE_ROOT} \
    --device          cuda \
    --batch_size      {BATCH_SIZE} \
    --train_vi_len    {TRAIN_VI_LEN} \
    --val_vi_len      {VAL_VI_LEN} \
    --num_epoch       {NUM_EPOCH} \
    --lr              {LR} \
    --lr_min          {LR_MIN} \
    --scheduler       {SCHEDULER} \
    --weight_decay    {WEIGHT_DECAY} \
    --tfr             {TFR} \
    --tfr_sde         {TFR_SDE} \
    --tfr_d_step      {TFR_D_STEP} \
    --tfr_d_period    {TFR_D_PERIOD} \
    --kl_anneal_type  {KL_ANNEAL_TYPE} \
    --kl_anneal_cycle {KL_ANNEAL_CYCLE} \
    --kl_anneal_ratio {KL_ANNEAL_RATIO} \
    --kl_max          {KL_MAX} \
    --per_save        {PER_SAVE} \
    --num_workers     {NUM_WORKERS} \
    {resume_flags}

## Optional — run the tester on the trained checkpoint

Generates `submission.csv` and `pred_seq*.gif` under `SAVE_ROOT`. Uses pure autoregressive 629-step rollout with z ∼ N(0, I).

In [ ]:
!python Tester.py \
    --DR        {DATASET_PATH} \
    --save_root {SAVE_ROOT} \
    --ckpt_path {SAVE_ROOT}/epoch=last.ckpt